# Lección 9: Grafos y Redes en Álgebra Lineal
### Álgebra Lineal Computacional y Fundamentos de la Física Matemática

---

Este cuaderno de Jupyter Colab aborda de manera formal y computacional la **Teoría de Grafos**. Los grafos son estructuras discretas utilizadas para modelar relaciones binarias entre objetos de un conjunto. Históricamente originados con Leonhard Euler y el famoso problema de los Puentes de Königsberg en 1736, los grafos son hoy en día la base del modelado de redes de telecomunicaciones, bases de datos no relacionales, análisis de redes sociales, estructuras moleculares en física y química, y algoritmos de optimización de rutas. Esta lección combina la fundamentación matemática rigurosa con la implementación práctica en Python mediante la biblioteca de análisis de redes `networkx` y cálculo matricial con `numpy`.


## Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Comprender** la definición matemática formal de un grafo $G(V, E)$ y su clasificación general (simples, multígrafos, pseudografos y dirigidos).
2. **Aplicar** los teoremas fundamentales de grado (Teorema de los Apretones de Manos para grafos dirigidos y no dirigidos).
3. **Representar** grafos utilizando listas de adyacencia, matrices de adyacencia y matrices de incidencia.
4. **Analizar** dinámicas de dominancia y liderazgo en grupos mediante potencias de matrices de adyacencia ($M^2$ y $M+M^2$).
5. **Identificar** y verificar isomorfismos de grafos a través de mapeos biyectivos y sus propiedades invariantes.
6. **Evaluar** la conectividad (conexo, fuertemente conexo y débilmente conexo) y simular recorridos sistemáticos en grafos (BFS y DFS).


## 1. Introducción y Clasificación de Grafos

Un **grafo** $G$ es una estructura matemática que consta de dos partes:
* Un conjunto $V = V(G)$ no vacío de elementos llamados **vértices (nodos o puntos)**.
* Un conjunto $E = E(G)$ de elementos llamados **aristas (enlaces o arcos)**, las cuales conectan pares de vértices.

Dependiendo de la naturaleza de las aristas, los grafos se clasifican formalmente en:

| Tipo | Aristas | ¿Admite aristas múltiples? | ¿Admite bucles? |
| :--- | :---: | :---: | :---: |
| **Grafo simple** | No dirigidas | No | No |
| **Multígrafo** | No dirigidas | Sí | No |
| **Pseudografo** | No dirigidas | Sí | Sí |
| **Grafo dirigido** | Dirigidas | No | Sí |
| **Multígrafo dirigido** | Dirigidas | Sí | Sí |

### 1.1 Tipos Especiales de Grafos
* **Grafo nulo:** Grafo sin vértices ni aristas ($V = \emptyset, E = \emptyset$).
* **Grafo vacío:** Grafo con vértices pero sin aristas ($E = \emptyset$).
* **Grafo trivial:** Grafo con un único vértice y sin aristas.
* **Grafo completo ($K_n$):** Grafo simple en el cual cada par de vértices distintos está conectado por una arista única. Posee exactamente $\frac{n(n-1)}{2}$ aristas.
* **Grafo bipartito:** Un grafo cuyo conjunto de vértices $V$ se puede particionar en dos conjuntos disjuntos $V_1$ y $V_2$, de modo que cada arista conecta un vértice de $V_1$ con uno de $V_2$ (no existen aristas internas en $V_1$ ni en $V_2$).
* **Grafo rueda ($W_n$):** Grafo formado al conectar un vértice central "eje" a todos los vértices de un ciclo simple de longitud $n-1$ ($C_{n-1}$).
* **Grafo plano:** Aquel que puede ser dibujado en el plano cartesiano sin que sus aristas se crucen.
* **Árbol:** Grafo no dirigido conexo y sin ciclos.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 1. Crear un Grafo Simple No Dirigido
G_simple = nx.Graph()
G_simple.add_nodes_from([1, 2, 3, 4])
G_simple.add_edges_from([(1, 2), (2, 3), (3, 4), (4, 1), (1, 3)])

# 2. Crear un Grafo Dirigido (Dígrafo)
G_dirigido = nx.DiGraph()
G_dirigido.add_nodes_from(['A', 'B', 'C'])
G_dirigido.add_edges_from([('A', 'B'), ('B', 'C'), ('C', 'A'), ('B', 'A')])

# Graficar ambos grafos
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Grafo Simple
pos_s = nx.spring_layout(G_simple, seed=42)
nx.draw(G_simple, pos_s, ax=axes[0], with_labels=True, node_color='lightblue', 
        node_size=800, font_weight='bold', edge_color='gray', width=2)
axes[0].set_title("Grafo No Dirigido Simple", fontsize=11, fontweight='bold')

# Grafo Dirigido
pos_d = nx.circular_layout(G_dirigido)
nx.draw(G_dirigido, pos_d, ax=axes[1], with_labels=True, node_color='lightgreen', 
        node_size=800, font_weight='bold', edge_color='black', width=2, 
        arrowsize=20, arrowstyle='->')
axes[1].set_title("Grafo Dirigido (Dígrafo)", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()



## 2. Relaciones de Vecindad y Teoremas de Grado

En un grafo no dirigido $G = (V, E)$:
* **Adyacencia:** Dos vértices $u, v \in V$ son adyacentes (o vecinos) si $\{u, v\} \in E$. Se denota como $u \sim v$.
* **Incidencia:** Una arista $e = \{u, v\}$ es incidente con los vértices $u$ y $v$.
* **Vecindad abierta:** $N(v) = \{u \in V \mid \{u, v\} \in E\}$ (no incluye a $v$).
* **Vecindad cerrada:** $N[v] = N(v) \cup \{v\}$.

### 2.1 Grado de un Vértice
El **grado** de un vértice $v$, denotado por $\delta(v)$ o $deg(v)$, es el número de aristas incidentes con él.
* **Nota sobre bucles:** En un pseudografo, un bucle en el vértice $v$ contribuye con **dos unidades** al grado $\delta(v)$, ya que la arista entra y sale del mismo vértice.

### 2.2 Teorema de los Apretones de Manos (Handshaking Lemma)
Para cualquier grafo no dirigido con $e$ aristas:
$$2e = \sum_{v \in V} \delta(v)$$

* **Corolario:** Todo grafo no dirigido tiene un **número par de vértices con grado impar**.

### 2.3 Grados en Grafos Dirigidos
En un dígrafo, cada vértice tiene:
* **Grado de entrada (in-degree) $\delta^-(v)$:** Número de aristas que tienen a $v$ como vértice final.
* **Grado de salida (out-degree) $\delta^+(v)$:** Número de aristas que tienen a $v$ como vértice inicial.
* **Teorema de la suma de grados dirigidos:**
  $$\sum_{v \in V} \delta^-(v) = \sum_{v \in V} \delta^+(v) = |E|$$
  (Un bucle suma 1 al grado de entrada y 1 al de salida).


In [ ]:
# Crear grafos especiales utilizando NetworkX
K_5 = nx.complete_graph(5)            # Grafo completo K_5
W_6 = nx.wheel_graph(6)               # Grafo rueda W_6
Bip = nx.complete_bipartite_graph(3, 3) # Grafo bipartito completo K_3,3

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# K_5
nx.draw_circular(K_5, ax=axes[0], with_labels=True, node_color='gold', node_size=600, font_weight='bold')
axes[0].set_title("Grafo Completo $K_5$", fontsize=11, fontweight='bold')

# W_6
nx.draw_kamada_kawai(W_6, ax=axes[1], with_labels=True, node_color='coral', node_size=600, font_weight='bold')
axes[1].set_title("Grafo Rueda $W_6$", fontsize=11, fontweight='bold')

# Bipartito K_3,3
# Para dibujar el bipartito, separamos las coordenadas en dos columnas
pos_bip = {}
for node in Bip.nodes():
    if node < 3: # Partición 1
        pos_bip[node] = (0, node)
    else:        # Partición 2
        pos_bip[node] = (1, node - 3)
nx.draw(Bip, pos_bip, ax=axes[2], with_labels=True, node_color='lightpink', node_size=600, font_weight='bold')
axes[2].set_title("Grafo Bipartito Completo $K_{3,3}$", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Verificación del Teorema de los Apretones de Manos en W_6
grados = [val for node, val in W_6.degree()]
num_aristas = W_6.number_of_edges()
suma_grados = sum(grados)

print("--- Verificación del Teorema de los Apretones de Manos (Rueda W_6) ---")
print(f"Número de vértices (n) = {W_6.number_of_nodes()}")
print(f"Grados individuales de los vértices: {grados}")
print(f"Suma de grados de los vértices = {suma_grados}")
print(f"2 * Número de aristas (2 * {num_aristas}) = {2 * num_aristas}")
print(f"¿Se cumple la igualdad sum(deg(v)) == 2*e? {suma_grados == 2 * num_aristas}")



## 3. Representación de Grafos mediante Álgebra Matricial

La representación visual de grafos puede ser computacionalmente ineficiente. Por ello, el Álgebra Lineal nos brinda herramientas formales de almacenamiento y cálculo matricial:

### 3.1 Lista de Adyacencia
Asocia a cada vértice $v$ una lista que contiene a todos sus vértices vecinos.

### 3.2 Matriz de Adyacencia ($A$)
Sea $G(V, E)$ un grafo simple con $|V| = n$. La matriz de adyacencia es una matriz booleana $A \in \mathbb{R}^{n \times n}$ cuyas componentes son:
$$a_{ij} = \begin{cases} 1 & \text{si } \{v_i, v_j\} \in E \quad (\text{o arista dirigida } v_i \to v_j) \\ 0 & \text{en caso contrario} \end{cases}$$

* **Propiedades:** En grafos no dirigidos, $A$ es simétrica ($A = A^T$). En dígrafos, no tiene por qué serlo.
* **Caminos de longitud $k$:** Si elevamos la matriz de adyacencia a la potencia $k$, el valor de la componente $(A^k)_{ij}$ representa el número exacto de caminos o recorridos de longitud $k$ que conectan el vértice $v_i$ con el vértice $v_j$.

### 3.3 Caso Práctico: Análisis de Dominancia y Liderazgo
Un psicólogo estudia las relaciones de dominancia/influencia en un grupo de trabajo compuesto por 5 personas: Aurelia ($A$), Berta ($B$), Carlos ($C$), Daniela ($D$), Esteban ($E$).
Mapeamos la influencia como un grafo dirigido donde $X \to Y$ significa que $X$ domina a $Y$. La matriz de adyacencia $M$ es:

$$M = \begin{pmatrix} 
0 & 1 & 1 & 0 & 1 \\ 
0 & 0 & 0 & 1 & 0 \\ 
0 & 1 & 0 & 0 & 1 \\ 
1 & 0 & 1 & 0 & 1 \\ 
1 & 1 & 0 & 0 & 0 
\end{pmatrix}$$

* **Liderazgo de primer orden:** Suma de las filas de $M$ (número de personas dominadas directamente).
* **Liderazgo de segundo orden ($M^2$):** Indica a cuántas personas puede dominar un integrante indirectamente a través de otro.
* **Puntaje de Influencia Total ($M + M^2$):** La suma de las filas de esta matriz total unifica la dominancia global. Declararemos al líder del grupo calculando estas potencias.

### 3.4 Matriz de Incidencia ($B$)
Relaciona los vértices con las aristas. Para un grafo no dirigido con $n$ vértices y $m$ aristas, es una matriz $B \in \mathbb{R}^{n \times m}$ definida por:
$$b_{ij} = \begin{cases} 1 & \text{si la arista } e_j \text{ es incidente con el vértice } v_i \\ 0 & \text{en caso contrario} \end{cases}$$

* **Propiedad:** Cada columna de la matriz de incidencia contiene exactamente dos '1's (o un solo '1' si la arista es un bucle).


In [ ]:
import numpy as np

# Definimos las variables de los nombres correspondientes a las filas/columnas
personas = ["Aurelia", "Berta", "Carlos", "Daniela", "Esteban"]
n = len(personas)

# Matriz de adyacencia de dominancia M
M = np.array([
    [0, 1, 1, 0, 1],  # A domina a B, C, E
    [0, 0, 0, 1, 0],  # B domina a D
    [0, 1, 0, 0, 1],  # C domina a B, E
    [1, 0, 1, 0, 1],  # D domina a A, C, E
    [1, 1, 0, 0, 0]   # E domina a A, B
], dtype=int)

# Cálculo de dominancia de segundo orden M^2
M_2 = np.linalg.matrix_power(M, 2)

# Cálculo de influencia global M + M^2
M_total = M + M_2

# Suma de filas para obtener la dominancia total
dominancia_1er = np.sum(M, axis=1)
dominancia_2do = np.sum(M_2, axis=1)
influencia_total = np.sum(M_total, axis=1)

# Imprimir resultados tabulados
print("-" * 75)
print(f"{'Persona':<12} | {'Dominancia 1er Orden':<20} | {'Dominancia 2do Orden':<20} | {'Influencia Total':<16}")
print("-" * 75)
for i in range(n):
    print(f"{personas[i]:<12} | {dominancia_1er[i]:^20} | {dominancia_2do[i]:^20} | {influencia_total[i]:^16}")
print("-" * 75)

# Identificar líder
lider_idx = np.argmax(influencia_total)
print(f"El líder indiscutible del grupo (mayor influencia total M + M^2) es: {personas[lider_idx]}")
print(f"Puntaje total de Daniela: {influencia_total[lider_idx]} (Dominancia directa: {dominancia_1er[lider_idx]}, Indirecta: {dominancia_2do[lider_idx]})")



In [ ]:
# Crear un grafo y calcular su matriz de incidencia de forma manual y usando NetworkX
G_incidence = nx.Graph()
G_incidence.add_nodes_from(['v1', 'v2', 'v3', 'v4'])
edges = [('v1', 'v2'), ('v2', 'v3'), ('v3', 'v4'), ('v4', 'v1'), ('v2', 'v4')]
G_incidence.add_edges_from(edges)

# Obtener la matriz de incidencia en formato denso
# sparse=False para obtener matriz numpy densa
B_matrix = nx.incidence_matrix(G_incidence, oriented=False).todense().astype(int)

# Mostrar la matriz de incidencia
print("--- Matriz de Incidencia del Grafo ---")
print("Vértices:", list(G_incidence.nodes()))
print("Aristas:", edges)
print("\nMatriz B:")
print(B_matrix)

# Verificación de propiedades de la matriz de incidencia
suma_columnas = np.sum(B_matrix, axis=0)
print(f"\nSuma de las columnas (debe ser 2 por arista): {suma_columnas.tolist()[0]}")
suma_filas = np.sum(B_matrix, axis=1)
print(f"Suma de las filas (grados de los vértices): {suma_filas.tolist()[0]}")
print(f"Grados obtenidos con degree(): {[val for node, val in G_incidence.degree()]}")



## 4. Grafos Isomorfos

Dos grafos simples $G_1 = (V_1, E_1)$ y $G_2 = (V_2, E_2)$ son **isomorfos** si existe una función biyectiva $f: V_1 \to V_2$ con la propiedad de que para cada par de vértices $u, v \in V_1$, $u$ y $v$ son adyacentes en $G_1$ si y sólo si $f(u)$ y $f(v)$ son adyacentes en $G_2$. La función $f$ se denomina **isomorfismo**.

### 4.1 Invariantes bajo Isomorfismo
Determinar si dos grafos grandes son isomorfos es un problema complejo (con n! biyecciones posibles). Sin embargo, si dos grafos son isomorfos, deben compartir ciertas propiedades invariantes:
1. Tener el mismo **número de vértices** ($|V_1| = |V_2|$).
2. Tener el mismo **número de aristas** ($|E_1| = |E_2|$).
3. Tener la misma **secuencia de grados** (ordenada de menor a mayor).

### 4.2 Ejemplo Práctico (Isomorfismo del Cubo 3D)
Representaremos dos grafos isomorfos:
* **Grafo G:** Dibujado como un grafo bipartito con las particiones de nodos $V_1 = \{a, b, c, d\}$ y $V_2 = \{g, h, i, j\}$.
* **Grafo H:** Dibujado en su forma clásica de un cubo en el plano con nodos del $1$ al $8$.

El isomorfismo exacto $f$ que mapea las adyacencias es:
$$f(a)=1, \quad f(b)=6, \quad f(c)=8, \quad f(d)=3$$
$$f(g)=5, \quad f(h)=2, \quad f(i)=4, \quad f(j)=7$$

Verificaremos este isomorfismo programáticamente y visualizaremos ambas estructuras lado a lado.


In [ ]:
# Definimos el Grafo G (Bipartito del cubo)
G_isom = nx.Graph()
# Vértices de G
G_isom.add_nodes_from(['a', 'b', 'c', 'd', 'g', 'h', 'i', 'j'])
# Aristas de G
edges_G = [
    ('a', 'h'), ('a', 'i'), ('a', 'g'),
    ('b', 'g'), ('b', 'j'), ('b', 'h'),
    ('c', 'j'), ('c', 'g'), ('c', 'i'),
    ('d', 'h'), ('d', 'i'), ('d', 'j')
]
G_isom.add_edges_from(edges_G)

# Definimos el Grafo H (Cubo 3D tradicional)
H_isom = nx.Graph()
H_isom.add_nodes_from([1, 2, 3, 4, 5, 6, 7, 8])
edges_H = [
    (1, 2), (2, 3), (3, 4), (4, 1), # Cara externa
    (5, 6), (6, 7), (7, 8), (8, 5), # Cara interna
    (1, 5), (2, 6), (3, 7), (4, 8)  # Conexiones transversales
]
H_isom.add_edges_from(edges_H)

# Comprobar isomorfismo con NetworkX
son_isomorfos = nx.is_isomorphic(G_isom, H_isom)
print(f"¿El Grafo G y el Grafo H son isomorfos? {son_isomorfos}")

# Definimos el mapeo del isomorfismo f dado en el PDF
mapping = {'a': 1, 'b': 6, 'c': 8, 'd': 3, 'g': 5, 'h': 2, 'i': 4, 'j': 7}

# Validamos que el mapeo mapee correctamente todas las aristas de G en H
mapeo_valido = True
for u, v in edges_G:
    mu, mv = mapping[u], mapping[v]
    if not H_isom.has_edge(mu, mv):
        mapeo_valido = False
        print(f"Fallo en arista: ({u},{v}) -> ({mu},{mv}) no existe en H")

print(f"¿El mapeo biyectivo manual propuesto en el PDF es matemáticamente válido? {mapeo_valido}")

# Visualización lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Dibujar G con estructura Bipartita
pos_G = {}
left_nodes = ['a', 'b', 'c', 'd']
right_nodes = ['g', 'h', 'i', 'j']
for idx, node in enumerate(left_nodes):
    pos_G[node] = (0, -idx)
for idx, node in enumerate(right_nodes):
    pos_G[node] = (1, -idx)
    
nx.draw(G_isom, pos_G, ax=axes[0], with_labels=True, node_color='lightblue', 
        node_size=700, font_weight='bold', edge_color='gray', width=2)
axes[0].set_title("Grafo G (Disposición Bipartita)", fontsize=11, fontweight='bold')

# Dibujar H con forma geométrica de Cubo 3D
pos_H = {
    1: (0, 2),  2: (2, 2),  3: (2, 0),  4: (0, 0),    # Cara exterior
    5: (0.5, 1.5), 6: (1.5, 1.5), 7: (1.5, 0.5), 8: (0.5, 0.5) # Cara interior
}
nx.draw(H_isom, pos_H, ax=axes[1], with_labels=True, node_color='lightgreen', 
        node_size=700, font_weight='bold', edge_color='black', width=2)
axes[1].set_title("Grafo H (Cubo 3D)", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()



## 5. Conectividad en Grafos y Recorridos Sistemáticos

### 5.1 Conectividad
* **Grafo conexo (no dirigido):** Existe un camino (trayectoria de vértices adyacentes) entre cualquier par de vértices del grafo.
* **Componentes conexas:** Subgrafos conexos maximales disjuntos de un grafo.
* **Grafos Dirigidos:**
  * **Fuertemente Conexo:** Existe un camino dirigido de $u \to v$ y de $v \to u$ para cualquier par de vértices $u, v \in V$.
  * **Débilmente Conexo:** El grafo es conexo si se ignora el sentido de las flechas (su grafo no orientado equivalente es conexo).

### 5.2 Recorridos en Grafos
Para buscar elementos u optimizar caminos se usan algoritmos de búsqueda sistemática:
1. **Búsqueda en Anchura (BFS - Breadth-First Search):** Explora los nodos por niveles (primero todos los vecinos inmediatos de la raíz, luego sus vecinos, etc.). Utiliza una estructura de datos de tipo **Cola (FIFO)**. Es ideal para hallar caminos mínimos en grafos no ponderados.
2. **Búsqueda en Profundidad (DFS - Depth-First Search):** Se adentra lo más posible a lo largo de cada rama antes de retroceder (backtracking). Utiliza una estructura de tipo **Pila (LIFO)**. Útil para detectar ciclos y ordenamiento topológico.


In [ ]:
# 1. Crear un dígrafo para verificar conectividad fuerte y débil
G_con = nx.DiGraph()
G_con.add_nodes_from([1, 2, 3, 4])
G_con.add_edges_from([(1, 2), (2, 3), (3, 1), (3, 4)])

# Graficar dígrafo
plt.figure(figsize=(6, 4))
pos_con = nx.spring_layout(G_con, seed=24)
nx.draw(G_con, pos_con, with_labels=True, node_color='lightyellow', node_size=600, 
        font_weight='bold', edge_color='purple', arrowsize=15, arrowstyle='->')
plt.title("Dígrafo de Prueba de Conectividad", fontsize=11, fontweight='bold')
plt.show()

# Evaluar conectividad
fuerte = nx.is_strongly_connected(G_con)
debil = nx.is_weakly_connected(G_con)

print("--- Evaluación de Conectividad en el Dígrafo ---")
print(f"¿El dígrafo es fuertemente conexo? {fuerte}")
print("Explicación: No, porque de 4 no se puede volver a ningún nodo.")
print(f"¿El dígrafo es débilmente conexo? {debil}")
print("Explicación: Sí, porque si quitamos las flechas, la estructura es conexa.")

# 2. Simulación de Recorridos BFS y DFS en un grafo no dirigido
G_search = nx.Graph()
G_search.add_edges_from([
    ('A', 'B'), ('A', 'C'),
    ('B', 'D'), ('B', 'E'),
    ('C', 'F'), ('E', 'F')
])

# Recorrido BFS desde la raíz 'A'
bfs_edges = list(nx.bfs_edges(G_search, source='A'))
bfs_nodes = ['A'] + [v for u, v in bfs_edges]

# Recorrido DFS desde la raíz 'A'
dfs_edges = list(nx.dfs_edges(G_search, source='A'))
dfs_nodes = ['A'] + [v for u, v in dfs_edges]

print("\n--- Simulación de Recorridos (Inicio en 'A') ---")
print(f"Vértices visitados en orden BFS: {bfs_nodes}")
print(f"Vértices visitados en orden DFS: {dfs_nodes}")



## 6. Resumen de la Lección

1. **Definición y Clasificación:** Un grafo es una estructura constituida por nodos (vértices) y enlaces (aristas). Se diferencian por el sentido del enlace (no dirigidos o dirigidos) y por admitir o no aristas múltiples y bucles.
2. **Teoremas de Grado:** El Teorema de los Apretones de Manos establece que la suma de grados es el doble del número de aristas. En dígrafos, la suma de grados de entrada es igual a la de salida y equivale al total de aristas.
3. **Representación Matricial:** Las matrices de adyacencia e incidencia facilitan el modelado matemático. Las potencias de la matriz de adyacencia ($M^k$) revelan la cantidad de caminos de longitud $k$ en la red.
4. **Dominancia y Liderazgo:** Mediante la matriz total de influencia $M + M^2$ es posible clasificar científicamente la influencia directa e indirecta de los agentes en una red jerárquica.
5. **Isomorfismo de Grafos:** Determina si dos representaciones visuales corresponden al mismo grafo estructural mediante bijecciones que preservan la adyacencia.
6. **Conectividad y Búsqueda:** Evaluamos si un grafo está unificado (conexo) o separado. Los algoritmos de búsqueda BFS (por niveles, usando colas) y DFS (profundo, usando pilas) permiten recorrer sistemáticamente las redes.

---

## 7. Referencias Bibliográficas

1. Rosen, K. H. (2004). *Matemática discreta y sus aplicaciones*. McGraw-Hill.
2. Lipschutz, S. (2009). *Matemáticas discretas*. Colección Schaum, McGraw-Hill.
3. García, M. (2020). *Teoría de Grafos y Estructuras de Datos*. Universidad Tecnológica de la Mixteca.
4. Wikipedia, La enciclopedia libre. *Teoría de grafos, Isomorfismo de grafos, Conectividad en grafos*.
